# CON_EP_RANK 因子

CON_EP_RANK 因子：未来12个月一致预期EP的申万二级行业内位序因子

## 原始因子指标计算

In [2]:
"""
BigQuant：未来12个月一致预期EP行业内位序因子的基础指标计算

基础指标：IC均值、RankIC均值、ICIR、RankICIR、因子收益率均值、
t值均值，以及IC/RankIC时序图。

说明：
1. 因子值使用统计日真实未复权收盘价，避免用复权价格计算估值比率。
2. 收益标签使用后复权收盘价计算“本截面至下一截面”的收益。
3. 这是因子研究诊断，不是可交易组合回测；不包含交易成本和成交约束。
4. 运行结果仅显示并保留在内存中，不自动保存CSV或图片。
5. BigQuant数据字段已按公开数据文档编写，但本文件未在BigQuant环境实跑。
"""

from __future__ import annotations

import time
import warnings
from dataclasses import dataclass
from datetime import timedelta
from typing import Iterable, List, Tuple, Union

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
from scipy import stats

import dai


# =========================
# 参数区
# =========================
START_DATE = "2025-05-09"
END_DATE = "2026-06-30"

# 支持 "W-FRI"（周频）、"M"（月频）、"Q"（季频），或正整数交易日间隔。
REBALANCE_FREQ: Union[str, int] = "M"

INDUSTRY_STANDARD = "sw2021"
MIN_INDUSTRY_SIZE = 10
MIN_CROSS_SECTION_SIZE = 30
MIN_LIST_DAYS = 90

# 为END_DATE之后预留下一截面；最终没有完整未来收益的截面会自动删除。
FORWARD_BUFFER_DAYS = 400

@dataclass(frozen=True)
class SignalPair:
    signal_date: pd.Timestamp
    return_end_date: pd.Timestamp


def log(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {message}")


def set_chinese_font() -> str:
    """选择环境中可用的中文字体，避免图表中文乱码。"""
    candidates = [
        "Microsoft YaHei",
        "SimHei",
        "Noto Sans CJK SC",
        "Source Han Sans CN",
        "WenQuanYi Micro Hei",
        "Noto Sans CJK JP",
        "PingFang SC",
        "Arial Unicode MS",
    ]
    available = {item.name for item in font_manager.fontManager.ttflist}
    selected = next((name for name in candidates if name in available), "DejaVu Sans")
    plt.rcParams["font.sans-serif"] = [selected]
    plt.rcParams["axes.unicode_minus"] = False
    if selected == "DejaVu Sans":
        warnings.warn("未找到常用中文字体，图表中文可能无法完整显示。")
    return selected


def load_factor_source(query_end_date: str) -> pd.DataFrame:
    """读取时点一致预期、真实价格、历史行业和风险状态。"""
    sql = f"""
    SELECT
        f.date,
        f.instrument,
        f.forecast_eps_12m,
        r.close AS real_close,
        i.industry_level2_code,
        i.industry_level2_name,
        p.is_risk_warning,
        p.list_days
    FROM cn_stock_financial_forecast_consensus_rolling AS f
    JOIN cn_stock_real_bar1d AS r
      ON f.date = r.date AND f.instrument = r.instrument
    JOIN cn_stock_industry_component AS i
      ON f.date = i.date AND f.instrument = i.instrument
    JOIN cn_stock_prefactors AS p
      ON f.date = p.date AND f.instrument = p.instrument
    WHERE i.industry = '{INDUSTRY_STANDARD}'
      AND p.is_risk_warning = 0
      AND p.list_days >= {MIN_LIST_DAYS}
      AND r.amount > 0
    """
    df = dai.query(
        sql,
        filters={"date": [START_DATE, query_end_date]},
    ).df()
    if df.empty:
        raise ValueError("一致预期查询结果为空，请检查日期、数据权限和字段可用性。")
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    return df


def load_adjusted_prices(query_end_date: str) -> pd.DataFrame:
    """读取后复权收盘价，仅用于计算跨截面收益标签。"""
    sql = """
    SELECT date, instrument, close AS adj_close
    FROM cn_stock_bar1d
    """
    df = dai.query(
        sql,
        filters={"date": [START_DATE, query_end_date]},
    ).df()
    if df.empty:
        raise ValueError("行情查询结果为空，请检查日期和数据权限。")
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    return df


def select_signal_dates(
    available_dates: Iterable[pd.Timestamp],
    frequency: Union[str, int],
) -> List[pd.Timestamp]:
    """从可用数据日期中选取每期最后一个截面。"""
    dates = pd.DatetimeIndex(sorted(pd.unique(list(available_dates))))
    if len(dates) == 0:
        return []

    if isinstance(frequency, int):
        if frequency <= 0:
            raise ValueError("整数调仓周期必须为正数。")
        return list(dates[::frequency])

    freq = str(frequency).upper()
    supported = {"W": "W-FRI", "W-FRI": "W-FRI", "M": "M", "Q": "Q"}
    if freq not in supported:
        raise ValueError("REBALANCE_FREQ仅支持W-FRI、M、Q或正整数。")

    date_series = pd.Series(dates, index=dates)
    period_key = dates.to_period(supported[freq])
    selected = date_series.groupby(period_key).max().sort_values()
    return list(pd.DatetimeIndex(selected.to_numpy()))


def make_signal_pairs(signal_dates: List[pd.Timestamp]) -> List[SignalPair]:
    start = pd.Timestamp(START_DATE)
    end = pd.Timestamp(END_DATE)
    pairs: List[SignalPair] = []
    for current_date, next_date in zip(signal_dates[:-1], signal_dates[1:]):
        if start <= current_date <= end:
            pairs.append(SignalPair(current_date, next_date))
    return pairs


def build_factor(panel: pd.DataFrame) -> pd.DataFrame:
    """计算EP原值和申万二级行业内中点百分位。"""
    df = panel.copy()
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(
        subset=["date", "instrument", "forecast_eps_12m", "real_close", "industry_level2_code"]
    )
    df = df[df["real_close"] > 0].copy()
    df["con_ep_12m"] = df["forecast_eps_12m"] / df["real_close"]

    group_cols = ["date", "industry_level2_code"]
    df["industry_sample_size"] = df.groupby(group_cols)["con_ep_12m"].transform("count")
    df = df[df["industry_sample_size"] >= MIN_INDUSTRY_SIZE].copy()

    rank = df.groupby(group_cols)["con_ep_12m"].rank(method="average", ascending=True)
    df["factor"] = (rank - 0.5) / df["industry_sample_size"]
    return df


def prepare_price_lookup(price_df: pd.DataFrame, dates: Iterable[pd.Timestamp]) -> pd.DataFrame:
    wanted_dates = set(pd.DatetimeIndex(dates))
    result = price_df[price_df["date"].isin(wanted_dates)].copy()
    return result.dropna(subset=["instrument", "adj_close"])


def calculate_one_cross_section(
    factor_df: pd.DataFrame,
    price_lookup: pd.DataFrame,
    pair: SignalPair,
) -> dict | None:
    exposure = factor_df.loc[
        factor_df["date"] == pair.signal_date,
        ["instrument", "factor"],
    ].copy()
    start_price = price_lookup.loc[
        price_lookup["date"] == pair.signal_date,
        ["instrument", "adj_close"],
    ].rename(columns={"adj_close": "start_close"})
    end_price = price_lookup.loc[
        price_lookup["date"] == pair.return_end_date,
        ["instrument", "adj_close"],
    ].rename(columns={"adj_close": "end_close"})

    sample = exposure.merge(start_price, on="instrument", how="inner")
    sample = sample.merge(end_price, on="instrument", how="inner")
    sample["future_return"] = sample["end_close"] / sample["start_close"] - 1.0
    sample = sample.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["factor", "future_return"]
    )

    if len(sample) < MIN_CROSS_SECTION_SIZE or sample["factor"].nunique() < 3:
        return None

    x = sample["factor"].astype(float)
    y = sample["future_return"].astype(float)
    factor_std = x.std(ddof=1)
    if not np.isfinite(factor_std) or factor_std <= 0:
        return None

    # 用下标取值以兼容不同版本的SciPy。
    ic = stats.pearsonr(x, y)[0]
    rank_ic = stats.spearmanr(x, y)[0]

    x_standardized = (x - x.mean()) / factor_std
    regression = stats.linregress(x_standardized, y)
    t_value = (
        regression.slope / regression.stderr
        if regression.stderr is not None and regression.stderr > 0
        else np.nan
    )

    return {
        "signal_date": pair.signal_date,
        "return_end_date": pair.return_end_date,
        "sample_count": len(sample),
        "ic": float(ic),
        "rank_ic": float(rank_ic),
        "factor_return": float(regression.slope),
        "t_value": float(t_value),
    }


def calculate_cross_section_series(
    factor_df: pd.DataFrame,
    price_df: pd.DataFrame,
    pairs: List[SignalPair],
) -> pd.DataFrame:
    if not pairs:
        raise ValueError("没有形成完整的信号期与下一收益期，请扩大日期范围。")

    required_dates = [p.signal_date for p in pairs] + [p.return_end_date for p in pairs]
    price_lookup = prepare_price_lookup(price_df, required_dates)
    records = []
    total = len(pairs)
    started = time.time()

    for index, pair in enumerate(pairs, start=1):
        record = calculate_one_cross_section(factor_df, price_lookup, pair)
        if record is not None:
            records.append(record)
        elapsed = time.time() - started
        log(
            f"截面计算 {index}/{total} ({index / total:.1%})，"
            f"日期={pair.signal_date.date()}，耗时={elapsed:.1f}秒"
        )

    result = pd.DataFrame(records)
    if result.empty:
        raise ValueError("所有截面均因样本不足或数据缺失被跳过。")
    return result.sort_values("signal_date").reset_index(drop=True)


def safe_ir(values: pd.Series) -> float:
    clean = values.dropna().astype(float)
    std = clean.std(ddof=1)
    if len(clean) < 2 or not np.isfinite(std) or std <= 0:
        return np.nan
    return float(clean.mean() / std)


def summarize(series_df: pd.DataFrame) -> pd.DataFrame:
    """按固定定义生成因子的基础指标。"""
    summary = {
        "因子名称": "CON_EP_12M_RANK",
        "开始日期": series_df["signal_date"].min().date().isoformat(),
        "结束日期": series_df["signal_date"].max().date().isoformat(),
        "截面周期": str(REBALANCE_FREQ),
        "有效截面数": int(len(series_df)),
        "平均截面样本数": float(series_df["sample_count"].mean()),
        "IC均值": float(series_df["ic"].mean()),
        "RankIC均值": float(series_df["rank_ic"].mean()),
        "ICIR": safe_ir(series_df["ic"]),
        "RankICIR": safe_ir(series_df["rank_ic"]),
        "因子收益率均值(每期,小数)": float(series_df["factor_return"].mean()),
        "t值均值": float(series_df["t_value"].mean()),
    }
    return pd.DataFrame([summary])


def plot_ic_series(series_df: pd.DataFrame) -> Tuple[plt.Figure, str]:
    font_name = set_chinese_font()
    fig, ax = plt.subplots(figsize=(12, 6.5), dpi=150)
    ax.plot(
        series_df["signal_date"],
        series_df["ic"],
        marker="o",
        linewidth=1.6,
        markersize=4,
        label="IC（Pearson）",
    )
    ax.plot(
        series_df["signal_date"],
        series_df["rank_ic"],
        marker="s",
        linewidth=1.6,
        markersize=4,
        label="RankIC（Spearman）",
    )
    ax.axhline(0, color="black", linewidth=0.9, alpha=0.65)
    ax.set_title("一致预期EP行业内位序因子：IC与RankIC时序")
    ax.set_xlabel("信号日期")
    ax.set_ylabel("相关系数")
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()
    plt.show()
    return fig, font_name


def main() -> Tuple[pd.DataFrame, pd.DataFrame, plt.Figure]:
    total_started = time.time()
    query_end = (pd.Timestamp(END_DATE) + timedelta(days=FORWARD_BUFFER_DAYS)).date().isoformat()

    log("阶段1/5：读取一致预期、真实价格、行业和风险状态")
    source = load_factor_source(query_end)
    log(
        f"因子源数据：{len(source):,}行，"
        f"{source['date'].min().date()}至{source['date'].max().date()}"
    )

    log("阶段2/5：构造未来12个月一致预期EP行业内中点百分位")
    factor_df = build_factor(source)
    coverage = factor_df.groupby("date")["instrument"].nunique()
    log(
        f"因子有效数据：{len(factor_df):,}行，"
        f"日均覆盖{coverage.mean():.0f}只股票"
    )

    signal_dates = select_signal_dates(factor_df["date"], REBALANCE_FREQ)
    pairs = make_signal_pairs(signal_dates)
    if not pairs:
        raise ValueError("指定区间内没有完整的截面收益标签。")

    log("阶段3/5：读取后复权价格并生成下一周期收益")
    price_df = load_adjusted_prices(query_end)

    log("阶段4/5：逐截面计算IC、RankIC、因子收益率和t值")
    series_df = calculate_cross_section_series(factor_df, price_df, pairs)
    summary_df = summarize(series_df)

    log("阶段5/5：绘制IC与RankIC时序图")
    figure, font_name = plot_ic_series(series_df)

    print("\n因子基础指标")
    print(summary_df.to_string(index=False))
    log(
        f"完成：有效截面{len(series_df)}个，中文字体={font_name}，"
        f"总耗时={time.time() - total_started:.1f}秒"
    )
    return summary_df, series_df, figure


if __name__ == "__main__":
    basic_metrics, cross_section_series, ic_rankic_figure = main()


[13:11:25] 阶段1/5：读取一致预期、真实价格、行业和风险状态


InvalidInputException: [31m<a href='https://bigquant.com/data/datasources/cn_stock_financial_forecast_consensus_rolling' target="_blank">cn_stock_financial_forecast_consensus_rolling</a>表当前仅能访问2023-01-03 00:00:00至2023-12-29 00:00:00的数据，想获取完整数据权限，请<a href="https://bigquant.com/spro?from=aistudio" target="_blank">开通BigQuant旗舰版</a>获取数据服务>>[0m